# 진짜 튜닝된 LightGBM(best_params.json) 5-Fold OOF 생성

지금까지 `blend_cache/oof_lgbm.npy`는 사실 기본 파라미터 버전(0.73925)이었습니다.
이 노트북은 main.py가 실제로 쓰는 `best_params.json`(Optuna 150trial 튜닝 결과)을
불러와서, main.py와 동일한 진짜 최종 모델의 OOF를 생성합니다.

torch 없이 LightGBM만 사용하므로 빠르게(몇 분 안에) 끝납니다.

**사전 준비**: `attempt file/best_params.json`이 있어야 합니다 (이미 있을 거예요).

## 1. 라이브러리, 설정, best_params 로드

In [4]:
import os
import json
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
RANDOM_STATE = 1004
CACHE_DIR = "blend_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

with open("best_params.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

# 파일 구조가 {"best_params": {...}} 형태로 감싸져 있을 수도 있어서 안전하게 처리
best_params = loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

print("불러온 best_params:")
print(json.dumps(best_params, indent=2, ensure_ascii=False))

불러온 best_params:
{
  "n_estimators": 2357,
  "learning_rate": 0.013668973845707234,
  "num_leaves": 159,
  "max_depth": 5,
  "min_child_samples": 83,
  "subsample": 0.908634361540321,
  "colsample_bytree": 0.7944647062780377,
  "reg_alpha": 6.529095336314542,
  "reg_lambda": 0.00029237017648296726,
  "min_split_gain": 0.4557413392057567
}


## 2. 데이터 전처리 (main.py와 동일)

In [5]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

df_le = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col].astype(str))

y = df_le[TARGET_COL].values.astype(np.float32)
X_lgbm = df_le.drop(columns=[TARGET_COL])

print(f"전처리 완료: {X_lgbm.shape}")

전처리 완료: (256351, 67)


## 3. 5-Fold OOF 생성 (진짜 튜닝된 파라미터)

In [6]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_lgbm_tuned = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_lgbm, y), start=1):
    m = LGBMClassifier(**best_params, random_state=RANDOM_STATE, verbose=-1)
    m.fit(X_lgbm.iloc[tr_idx], y[tr_idx])
    oof_lgbm_tuned[val_idx] = m.predict_proba(X_lgbm.iloc[val_idx])[:, 1]
    fold_auc = roc_auc_score(y[val_idx], oof_lgbm_tuned[val_idx])
    print(f"Fold {fold}/{N_SPLITS}  LightGBM(튜닝) AUC: {fold_auc:.5f}")

score_tuned = roc_auc_score(y, oof_lgbm_tuned)
print(f"\nLightGBM(튜닝) 5-Fold 전체 OOF AUC: {score_tuned:.5f}")
print(f"(main.py 보고된 점수 0.7400과 비교: {score_tuned - 0.7400:+.5f})")

Fold 1/5  LightGBM(튜닝) AUC: 0.74396
Fold 2/5  LightGBM(튜닝) AUC: 0.74037
Fold 3/5  LightGBM(튜닝) AUC: 0.73804
Fold 4/5  LightGBM(튜닝) AUC: 0.73847
Fold 5/5  LightGBM(튜닝) AUC: 0.73926

LightGBM(튜닝) 5-Fold 전체 OOF AUC: 0.73998
(main.py 보고된 점수 0.7400과 비교: -0.00002)


## 4. 결과 저장

In [7]:
np.save(f"{CACHE_DIR}/oof_lgbm_tuned.npy", oof_lgbm_tuned)
np.save(f"{CACHE_DIR}/oof_y.npy", y)
print("저장 완료: blend_cache/oof_lgbm_tuned.npy")
print("\n참고: 기존 blend_cache/oof_lgbm.npy(미튜닝, 0.73925)는 그대로 두었습니다.")
print("이제부터 비교/블렌딩에는 oof_lgbm_tuned.npy를 쓰시면 됩니다.")

저장 완료: blend_cache/oof_lgbm_tuned.npy

참고: 기존 blend_cache/oof_lgbm.npy(미튜닝, 0.73925)는 그대로 두었습니다.
이제부터 비교/블렌딩에는 oof_lgbm_tuned.npy를 쓰시면 됩니다.
